### Speechinteraction with BERT

Download model from https://huggingface.co/docs/transformers/model_doc/mobilebert

Data processing: https://huggingface.co/docs/transformers/tasks/token_classification

Explanations to metrics: https://www.sciencedirect.com/science/article/pii/S1574954124002516

Token classification: https://huggingface.co/docs/transformers/tasks/token_classification

In [ ]:
# install necessary libraries
!pip install datasets
!pip install tokenizers
!pip install transformers
!pip install wandb
!pip install seqeval

In [2]:
# load dataset
from datasets import load_dataset

dataset = load_dataset("wikiann", "bn")

# INFO: You can also directly download pretrained models
#   from transformers import BertConfig, BertModel
# if model is on hugging face Hub
#   model = BertModel.from_pretrained("bert-base-uncased")
# from local folder
#   model = BertModel.from_pretrained("./test/saved_model/")

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [4]:
label_names = dataset["train"].features["ner_tags"].feature.names
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

In [8]:
# use autotokenizer
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

# choose from multiple different tokenizers
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
#tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")


#model = AutoModelForTokenClassification.from_pretrained(
#    "distilbert/distilbert-base-uncased", num_labels=13, id2label=id2label, label2id=label2id)

In [ ]:
testpharse = "Last week Gandalf visited the Shire"

In [9]:
#Get the values for input_ids, token_type_ids, attention_mask
def tokenize_adjust_labels(all_samples_per_split):
  tokenized_samples = tokenizer.batch_encode_plus(all_samples_per_split["tokens"], is_split_into_words=True)
  #tokenized_samples is not a datasets object so this alone won't work with Trainer API, hence map is used 
  #so the new keys [input_ids, labels (after adjustment)]
  #can be added to the datasets dict for each train test validation split
  total_adjusted_labels = []
  print(len(tokenized_samples["input_ids"]))
  for k in range(0, len(tokenized_samples["input_ids"])):
    prev_wid = -1
    word_ids_list = tokenized_samples.word_ids(batch_index=k)
    existing_label_ids = all_samples_per_split["ner_tags"][k]
    i = -1
    adjusted_label_ids = []
   
    for wid in word_ids_list:
      if(wid is None):
        adjusted_label_ids.append(-100)
      elif(wid!=prev_wid):
        i = i + 1
        adjusted_label_ids.append(existing_label_ids[i])
        prev_wid = wid
      else:
        label_name = label_names[existing_label_ids[i]]
        adjusted_label_ids.append(existing_label_ids[i])
        
    total_adjusted_labels.append(adjusted_label_ids)
  tokenized_samples["labels"] = total_adjusted_labels
  return tokenized_samples

tokenized_dataset = dataset.map(tokenize_adjust_labels, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

1000


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

1000


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

1000
1000
1000
1000
1000
1000
1000
1000
1000
1000


In [10]:
# Pad tokens for BERT
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

### WANDB 
Setup login & Projekt to inspect training values (Memory Usage, 
Framework to inspect weights and biases: https://wandb.ai/site

In [12]:
import os
import wandb
os.environ["WANDB_API_KEY"]=""      # insert your API Key here
os.environ["WANDB_ENTITY"]=""       # insert user name
os.environ["WANDB_PROJECT"]=""      #insert project name

In [ ]:
### Trainer API
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np
from datasets import load_metric


metric = load_metric("seqeval")
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    flattened_results = {
        "overall_precision": results["overall_precision"],
        "overall_recall": results["overall_recall"],
        "overall_f1": results["overall_f1"],
        "overall_accuracy": results["overall_accuracy"],
    }
    for k in results.keys():
      if(k not in flattened_results.keys()):
        flattened_results[k+"_f1"]=results[k]["f1"]

    return flattened_results

### Model training / fine tuning
Next, we load the chosen model, set our training parameters and use the Trainer to fine tune the model.

All metrics will be locked with wandb. 
For runtime-memory usage, look at WandB. This works for training, fine-tuning and normal runtime usage. 

For memory useage of the system, download the model and inspect the filesize;

### ! The output result was left inside the cell for you to remember the "Tutorial" an further inspect the data. - if you re-run the cell, the data will be lost !

You can also just copy the code and re-run it in a new cell

In [20]:
model = AutoModelForTokenClassification.from_pretrained("bert-base-multilingual-cased", num_labels=len(label_names))
training_args = TrainingArguments(
    output_dir="./fine_tune_bert_output",
    evaluation_strategy="steps",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=7,
    weight_decay=0.01,
    logging_steps = 1000,
    report_to="wandb",
    run_name = "ep_10_tokenized_11",
    save_strategy='no'     # no, epochs or steps
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Save model
# trainer.save_model("path_to_save")

trainer.train()
wandb.finish()

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

C:\Users\aless\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:148: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aless\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\aless\anaconda3\Lib\site-packages\transformers\training_args.py:1463: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: Currently logged in as: alessa-angerschmid. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,Loc F1,Org F1,Per F1
1000,0.707800,0.299627,0.865216,0.854271,0.859709,0.900367,0.847129,0.881561,0.852562
2000,0.271600,0.277548,0.895932,0.889101,0.892503,0.925209,0.873667,0.907422,0.900709
3000,0.156800,0.244004,0.923412,0.917172,0.920282,0.941912,0.900611,0.941734,0.923121
4000,0.090100,0.275168,0.908109,0.917865,0.912961,0.938614,0.895055,0.925014,0.923298


eval/LOC_f1,▁▄█▇
eval/ORG_f1,▁▄█▆
eval/PER_f1,▁▆██
eval/loss,█▅▁▅
eval/overall_accuracy,▁▅█▇
eval/overall_f1,▁▅█▇
eval/overall_precision,▁▅█▆
eval/overall_recall,▁▅██
eval/runtime,▅▁█▅
eval/samples_per_second,▄█▁▄
eval/steps_per_second,▄█▁▄
